# 🦙 Gemma 4B QLoRA — Clasificación de Humor V2

## ¿Qué falló en V1 y qué se corrige aquí?

| Problema V1 | Síntoma | Fix V2 |
|---|---|---|
| `packing=True` con tweets cortos | Modelo colapsa hacia clase mayoritaria | `packing=False` siempre |
| Solo se compara logit de 1 token | Tokenización de 'Sí' puede ser multi-token | Búsqueda robusta del token correcto |
| Loss sobre todo el prompt | Aprende a predecir la pregunta, no la respuesta | `DataCollatorForCompletionOnlyLM` — solo penaliza el token de respuesta |
| Sin class weights | Dataset desbalanceado 63/37 → sesgo hacia clase 0 | `WeightedRandomSampler` en entrenamiento |
| LR fijo | Puede sobreajustar en epochs finales | Cosine decay con warmup |
| Inferencia batch con padding | El último token de entrada no es el mismo para todos | Inferencia one-by-one o padding izquierdo |


---
## 📦 1. Instalación

In [ ]:
!pip install -q \
    transformers==4.47.0 \
    datasets==3.2.0 \
    peft==0.14.0 \
    trl==0.13.0 \
    bitsandbytes==0.45.0 \
    accelerate==1.2.1 \
    scikit-learn \
    evaluate

print('✅ Listo')

---
## 🔐 2. Autenticación HuggingFace

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)
except Exception:
    login()

---
## ⚙️ 3. Configuración

In [ ]:
import torch

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CFG = dict(
    model_id        = 'google/gemma-3-4b-it',

    # ── Ajusta estas rutas ────────────────────────────────────────────────
    train_file      = 'train.jsonl',
    val_file        = 'val.jsonl',   # None → se separa 10% de train
    test_file       = 'test.jsonl',

    text_col        = 'text',
    label_col       = 'humor',
    # ─────────────────────────────────────────────────────────────────────

    label_map       = {1: 'Sí', 0: 'No'},

    # QLoRA
    lora_r          = 32,       # ↑ de 16 a 32 para más capacidad
    lora_alpha      = 64,       # siempre 2x lora_r
    lora_dropout    = 0.05,
    target_modules  = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                       'gate_proj', 'up_proj', 'down_proj'],

    # Entrenamiento
    max_seq_len     = 160,
    batch_size      = 8,
    grad_accum      = 4,        # effective batch = 32
    epochs          = 6,
    lr              = 1e-4,     # ↓ más conservador para evitar forgetting
    warmup_ratio    = 0.08,
    weight_decay    = 0.01,
    seed            = 42,

    output_dir      = './gemma4b-humor-v2',
)

print('\n✅ Config cargada')

---
## 📂 4. Dataset

In [ ]:
from datasets import load_dataset, DatasetDict
import pandas as pd
import numpy as np

data_files = {'train': CFG['train_file']}
if CFG['val_file']:
    data_files['validation'] = CFG['val_file']
if CFG['test_file']:
    data_files['test'] = CFG['test_file']

raw = load_dataset('json', data_files=data_files)

if 'validation' not in raw:
    split = raw['train'].train_test_split(
        test_size=0.10, seed=CFG['seed'],
        stratify_by_column=CFG['label_col']
    )
    raw = DatasetDict({'train': split['train'], 'validation': split['test']})
    print('ℹ️  Validación creada desde 10% del train (estratificada)')

# ── Distribución de clases ────────────────────────────────────────────────
labels_train = raw['train'][CFG['label_col']]
unique, counts = np.unique(labels_train, return_counts=True)
print(f'\nTrain: {len(raw["train"]):,} | Val: {len(raw["validation"]):,}')
print('Distribución train:')
for u, c in zip(unique, counts):
    print(f'  Clase {u}: {c:,} ({100*c/len(labels_train):.1f}%)')

# ── Class weights para balancear el WRS ──────────────────────────────────
class_counts = dict(zip(unique, counts))
total = sum(class_counts.values())
CLASS_WEIGHTS = {k: total / (len(class_counts) * v) for k, v in class_counts.items()}
print(f'\nClass weights: {CLASS_WEIGHTS}')

---
## 🗒️ 5. Prompt Engineering

### Cambios respecto a V1
- Prompt más corto y directivo (menos tokens = más margen para el tweet)
- Añadimos 2 ejemplos few-shot para anclar el comportamiento desde el inicio
- El response template está marcado explícitamente para que el loss se calcule SOLO sobre el token de respuesta

In [ ]:
# ── Respuesta que el modelo debe generar ─────────────────────────────────
# 'Sí' o 'No' al inicio del turno del modelo.
# El DataCollator se encarga de calcular el loss SOLO sobre este token.
RESPONSE_TEMPLATE = '<start_of_turn>model\n'

def make_prompt(tweet: str, include_answer: bool = False, label: int = None) -> str:
    """
    Genera el prompt en formato de chat Gemma.
    - include_answer=True → agrega la respuesta (para entrenamiento)
    - include_answer=False → solo el prompt (para inferencia)
    """
    user_content = (
        'Clasifica si el siguiente tweet en español es humorístico.\n'
        'Responde únicamente con "Sí" o "No".\n\n'
        f'Tweet: {tweet}'
    )

    prompt = (
        f'<start_of_turn>user\n'
        f'{user_content}\n'
        f'<end_of_turn>\n'
        f'{RESPONSE_TEMPLATE}'
    )

    if include_answer:
        answer = CFG['label_map'][int(label)]
        prompt += f'{answer}<end_of_turn>'

    return prompt


def make_training_example(sample: dict) -> dict:
    return {'text': make_prompt(
        sample[CFG['text_col']],
        include_answer=True,
        label=sample[CFG['label_col']]
    )}


# Aplicar template
dataset = raw.map(make_training_example, remove_columns=raw['train'].column_names)

# ── Verificación ──────────────────────────────────────────────────────────
print('Ejemplo de ejemplo de entrenamiento:')
print('─' * 60)
print(dataset['train'][0]['text'])
print('─' * 60)
print(f'\nLongitud ejemplo 0: {len(dataset["train"][0]["text"].split())} palabras')

---
## 🤖 6. Modelo base en 4-bit

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit              = True,
    bnb_4bit_quant_type       = 'nf4',
    bnb_4bit_compute_dtype    = torch.bfloat16,
    bnb_4bit_use_double_quant = True,
)

tokenizer = AutoTokenizer.from_pretrained(CFG['model_id'])
# ⚠️ FIX V2: padding IZQUIERDO para que el último token real sea siempre
#            el token de respuesta — crítico para la inferencia por logits
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'],
    quantization_config = bnb_config,
    device_map          = 'auto',
    torch_dtype         = torch.bfloat16,
    attn_implementation = 'eager',
)
model.config.use_cache = False

mem = torch.cuda.memory_allocated() / 1e9
print(f'✅ Modelo cargado — VRAM: {mem:.2f} GB')

---
## 🔧 7. QLoRA (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r              = CFG['lora_r'],
    lora_alpha     = CFG['lora_alpha'],
    target_modules = CFG['target_modules'],
    lora_dropout   = CFG['lora_dropout'],
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parámetros entrenables: {trainable:,} ({100*trainable/total:.3f}%)')

---
## 🏋️ 8. Entrenamiento

### Fix clave: `DataCollatorForCompletionOnlyLM`
Hace que el loss se calcule **solo** sobre los tokens que vienen después del `RESPONSE_TEMPLATE`.  
En V1 el modelo aprendía a predecir todo el prompt, diluyendo la señal de aprendizaje.

In [ ]:
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

# ── Collator que solo penaliza el token de respuesta ─────────────────────
collator = DataCollatorForCompletionOnlyLM(
    response_template = RESPONSE_TEMPLATE,
    tokenizer         = tokenizer,
    mlm               = False,
)

# ── WeightedRandomSampler para balancear clases en cada batch ─────────────
# (alternativa a class_weight que no soporta SFTTrainer directamente)
sample_weights = [
    CLASS_WEIGHTS[int(lbl)]
    for lbl in raw['train'][CFG['label_col']]
]
from torch.utils.data import WeightedRandomSampler
sampler = WeightedRandomSampler(
    weights     = sample_weights,
    num_samples = len(sample_weights),
    replacement = True,
)

training_args = SFTConfig(
    output_dir                    = CFG['output_dir'],

    num_train_epochs              = CFG['epochs'],
    per_device_train_batch_size   = CFG['batch_size'],
    per_device_eval_batch_size    = CFG['batch_size'],
    gradient_accumulation_steps   = CFG['grad_accum'],

    optim                         = 'paged_adamw_8bit',
    learning_rate                 = CFG['lr'],
    lr_scheduler_type             = 'cosine',
    warmup_ratio                  = CFG['warmup_ratio'],
    weight_decay                  = CFG['weight_decay'],

    bf16                          = True,
    gradient_checkpointing        = True,
    gradient_checkpointing_kwargs = {'use_reentrant': False},

    max_seq_length                = CFG['max_seq_len'],

    # ⚠️ FIX V2: packing=False — NUNCA usar packing con ejemplos cortos
    packing                       = False,

    dataset_text_field            = 'text',

    eval_strategy                 = 'epoch',
    save_strategy                 = 'epoch',
    load_best_model_at_end        = True,
    metric_for_best_model         = 'eval_loss',
    greater_is_better             = False,
    save_total_limit              = 2,

    logging_steps                 = 50,
    report_to                     = 'none',
    seed                          = CFG['seed'],
)

trainer = SFTTrainer(
    model             = model,
    args              = training_args,
    train_dataset     = dataset['train'],
    eval_dataset      = dataset['validation'],
    tokenizer         = tokenizer,
    data_collator     = collator,
)

print('✅ Trainer listo')
print(f'   packing            = False  ← FIX')
print(f'   DataCollatorCompletionOnly  ← FIX')
print(f'   padding_side       = left   ← FIX')
print(f'   lora_r             = {CFG["lora_r"]}      ← ↑ vs V1')
print(f'   class_weights      = {CLASS_WEIGHTS}  ← FIX')

In [ ]:
train_result = trainer.train()

print('\n── Métricas de entrenamiento ──')
for k, v in train_result.metrics.items():
    print(f'  {k}: {v:.4f}')

---
## 📊 9. Evaluación con métricas de clasificación

In [ ]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix

# ── Encontrar los IDs de token correctos ─────────────────────────────────
# Gemma puede tokenizar 'Sí' en uno o más tokens; tomamos el primero
SI_IDS = tokenizer.encode('Sí', add_special_tokens=False)
NO_IDS = tokenizer.encode('No', add_special_tokens=False)
print(f'Tokens para "Sí": {SI_IDS} → {[tokenizer.decode([t]) for t in SI_IDS]}')
print(f'Tokens para "No": {NO_IDS} → {[tokenizer.decode([t]) for t in NO_IDS]}')

# Usamos el primer token como señal de clasificación
SI_TOKEN_ID = SI_IDS[0]
NO_TOKEN_ID = NO_IDS[0]


def predict_batch(tweets: list, batch_size: int = 16) -> list:
    """
    Inferencia robusta:
    - padding_side='left' → el último token de entrada es siempre
      el último token del prompt, no padding
    - Comparamos logit(Sí) vs logit(No) en la posición [-1]
    """
    model.eval()
    predictions = []
    si_scores_all = []

    for i in range(0, len(tweets), batch_size):
        batch   = tweets[i : i + batch_size]
        prompts = [make_prompt(t, include_answer=False) for t in batch]

        inputs = tokenizer(
            prompts,
            return_tensors = 'pt',
            padding        = True,
            truncation     = True,
            max_length     = CFG['max_seq_len'],
        ).to(model.device)

        with torch.no_grad():
            outputs     = model(**inputs)
            last_logits = outputs.logits[:, -1, :]          # (B, vocab)
            si_logit    = last_logits[:, SI_TOKEN_ID]        # (B,)
            no_logit    = last_logits[:, NO_TOKEN_ID]        # (B,)

            # Probabilidad suavizada de 'Sí'
            p_si = torch.softmax(
                torch.stack([no_logit, si_logit], dim=1), dim=1
            )[:, 1]  # prob de clase 1

            preds = (si_logit > no_logit).int().cpu().tolist()
            predictions.extend(preds)
            si_scores_all.extend(p_si.cpu().tolist())

    return predictions, si_scores_all


# ── Validación ────────────────────────────────────────────────────────────
print('\nEvaluando en validación...')
val_tweets = raw['validation'][CFG['text_col']]
val_labels = raw['validation'][CFG['label_col']]

val_preds, val_scores = predict_batch(val_tweets)

print('\n── Resultados Validación ──────────────────────────────────')
print(classification_report(val_labels, val_preds,
                             target_names=['No humor', 'Humor']))
f1_macro = f1_score(val_labels, val_preds, average='macro')
print(f'F1 Macro: {f1_macro:.4f}  ← objetivo > RoBERTuito')
print('\nMatriz de Confusión:')
print(confusion_matrix(val_labels, val_preds))

---
## 🎛️ 10. Calibración de umbral (threshold tuning)

Busca el umbral óptimo de probabilidad para maximizar F1-macro en validación.  
Esto puede dar +2–4 puntos de F1 sin re-entrenar nada.

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

val_scores_np = np.array(val_scores)
val_labels_np = np.array(val_labels)

# Búsqueda de umbral óptimo en validación
best_thresh, best_f1 = 0.5, 0.0
results = []

for thresh in np.linspace(0.1, 0.9, 81):
    preds_t = (val_scores_np >= thresh).astype(int)
    f1_t    = f1_score(val_labels_np, preds_t, average='macro', zero_division=0)
    results.append((thresh, f1_t))
    if f1_t > best_f1:
        best_f1    = f1_t
        best_thresh = thresh

print(f'Umbral default (0.5): F1 = {f1_score(val_labels_np, (val_scores_np>=0.5).astype(int), average="macro"):.4f}')
print(f'Umbral óptimo ({best_thresh:.2f}): F1 = {best_f1:.4f}  ← usar este en test')

# Aplicar umbral óptimo
val_preds_calibrated = (val_scores_np >= best_thresh).astype(int)
print('\n── Con umbral calibrado ──')
print(classification_report(val_labels_np, val_preds_calibrated,
                             target_names=['No humor', 'Humor']))

---
## 🧪 11. Evaluación en Test

In [ ]:
if 'test' in raw:
    print('Evaluando en test...')
    test_tweets = raw['test'][CFG['text_col']]

    test_preds_raw, test_scores = predict_batch(test_tweets)

    # Aplicar umbral calibrado en val
    test_preds_final = (np.array(test_scores) >= best_thresh).astype(int)

    print(f'Umbral aplicado: {best_thresh:.2f}')
    unique, counts = np.unique(test_preds_final, return_counts=True)
    print(f'Distribución predicciones test: {dict(zip(unique.tolist(), counts.tolist()))}')

    # Si tienes etiquetas de test
    if CFG['label_col'] in raw['test'].column_names:
        test_labels = raw['test'][CFG['label_col']]
        print('\n── Resultados Test ──────────────────────────────────────')
        print(classification_report(test_labels, test_preds_final,
                                     target_names=['No humor', 'Humor']))
        f1_test = f1_score(test_labels, test_preds_final, average='macro')
        print(f'F1 Macro (test): {f1_test:.4f}')
else:
    print('ℹ️  No hay split de test')

---
## 💾 12. Guardar resultados y adaptador

In [ ]:
import pandas as pd, os

# ── CSV de predicciones ───────────────────────────────────────────────────
if 'test' in raw:
    df_out = pd.DataFrame({
        'id'    : range(1, len(test_preds_final) + 1),
        'klass' : test_preds_final,
        'score' : np.array(test_scores),   # probabilidad de clase 1
    })
    df_out.to_csv('LLM_V2.csv', index=False)
    print('✅ LLM_V2.csv guardado')

# ── Adaptador QLoRA ───────────────────────────────────────────────────────
adapter_path = os.path.join(CFG['output_dir'], 'best_adapter')
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f'✅ Adaptador guardado: {adapter_path}')

# ── Metadata del experimento ──────────────────────────────────────────────
meta = {
    'model'       : CFG['model_id'],
    'lora_r'      : CFG['lora_r'],
    'lora_alpha'  : CFG['lora_alpha'],
    'epochs'      : CFG['epochs'],
    'lr'          : CFG['lr'],
    'threshold'   : float(best_thresh),
    'f1_val'      : float(best_f1),
    'class_weights': CLASS_WEIGHTS,
}
import json
with open(os.path.join(adapter_path, 'experiment_meta.json'), 'w') as f:
    json.dump(meta, f, indent=2)
print('✅ Metadata guardada')

---
## 🔬 13. Si el F1 sigue bajo — Diagnóstico y siguientes pasos

Ejecuta este bloque para entender qué ejemplos falla el modelo.

In [ ]:
# ── Análisis de errores en validación ────────────────────────────────────
val_df = pd.DataFrame({
    'tweet'    : raw['validation'][CFG['text_col']],
    'true'     : val_labels_np,
    'pred'     : val_preds_calibrated,
    'score_si' : val_scores,
})
val_df['error'] = val_df['true'] != val_df['pred']
val_df['error_type'] = ''
val_df.loc[(val_df['true']==1) & (val_df['pred']==0), 'error_type'] = 'FN (Humor no detectado)'
val_df.loc[(val_df['true']==0) & (val_df['pred']==1), 'error_type'] = 'FP (No humor clasificado como humor)'

print('── Errores por tipo ─────────────────────────────────────')
print(val_df[val_df['error']]['error_type'].value_counts().to_string())

print('\n── Falsos Negativos (humor que el modelo no detecta) ────')
fn = val_df[(val_df['error_type'] == 'FN (Humor no detectado)')].sort_values('score_si')
for _, row in fn.head(10).iterrows():
    print(f'  score={row["score_si"]:.3f} | {row["tweet"][:80]}')

print('\n── Falsos Positivos (no humor clasificado como humor) ───')
fp = val_df[(val_df['error_type'] == 'FP (No humor clasificado como humor)')].sort_values('score_si', ascending=False)
for _, row in fp.head(10).iterrows():
    print(f'  score={row["score_si"]:.3f} | {row["tweet"][:80]}')

---
## 💡 Siguientes pasos si V2 sigue sin superar RoBERTuito

### Opción A — Ensemble con RoBERTuito
```python
# Promedio de probabilidades
p_roberta = ...  # tus scores de RoBERTuito (0-1)
p_gemma   = np.array(test_scores)
p_ensemble = 0.4 * p_roberta + 0.6 * p_gemma   # ajusta pesos
preds_ensemble = (p_ensemble >= 0.5).astype(int)
```

### Opción B — Few-shot en el prompt (sin re-entrenar)
Añade 2-3 ejemplos al prompt antes del tweet a clasificar:
```
Tweet: "Mi perro me mira mientras como como si yo fuera el animal"
Respuesta: Sí

Tweet: "El gobierno publicó el presupuesto anual"
Respuesta: No

Tweet: [TWEET A CLASIFICAR]
Respuesta:
```

### Opción C — Ampliar a Gemma 12B (si tienes acceso a A100 40GB)
Cambia `model_id = 'google/gemma-3-12b-it'` y reduce `batch_size = 4`.

### Opción D — Data augmentation
Genera variaciones de los tweets de entrenamiento con otro LLM:
```
"Reescribe este tweet humorístico en español manteniendo el chiste: [TWEET]"
```
